In [ ]:
using Pkg
Pkg.activate(".")

In [ ]:
include("itg_instability.jl")

In [ ]:
using HDF5, Statistics, CairoMakie, FFTW

In [ ]:
function readRho(path)
    output = h5open("output/rho.h5","r"; swmr=true)
    ks = sort(filter(k -> occursin(r"^rho_\d+$", k), keys(output)), by = k -> parse(Int, split(k, "_")[2]))
    rho = map(x->Array(output[x]), ks);
    close(output)
    rho
end

In [ ]:
rho = readRho("output/rho.h5")

In [ ]:
drho = map(x -> (x.-mean(x)), rho);
drho2 = map(x -> mean(x .^2), drho);

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1]; xlabel="snapshot index", ylabel="⟨δρ²⟩",
          yscale=log10, title="Total fluctuation energy vs time")
lines!(ax, simTime[1:nDiag:nDiag*length(rho)], drho2)
fig

In [ ]:
frho = hcat(map(x->abs.(fft(x))[1,:,2], rho)...)

In [ ]:
# Grid parameters — must match itg_instability.jl

Nmodes = 8

dt_save = nDiag * simTime.dt          # nDiag × dt (time units between saved snapshots)

t = simTime[1:nDiag:nDiag*length(rho)]

# ky values corresponding to FFTW output indices 1 … Ny÷2+1
ky_vals = [2π / Ly * j for j in 0:Nmodes]

fig = Figure(size=(900, 500))
ax = Axis(fig[1, 1]; xlabel="t [ω_ci⁻¹]", ylabel="|ρ̂(kx=0, ky, kz₁)|",
          yscale=log10, title="Mode amplitudes: kx=0, nkz=1")
for i in 2:Nmodes                         # positive ky modes only, skip DC (i=1)
    lines!(ax, t, frho[i, :], label="ky = $(round(ky_vals[i], digits=2))")
end
Legend(fig[1, 2], ax, "ky")
fig

In [ ]:
# Fit growth rate γ in the linear phase: log|ρ̂(ky, t)| = γ(ky)⋅t + const
# Adjust t_fit_start / t_fit_end by inspecting the mode plot above
t_fit_start = 1200
t_fit_end   = 1500

i_start = findfirst(t .>= t_fit_start)
i_end   = findfirst(t .>= t_fit_end)
t_fit   = t[i_start:i_end]
X_fit   = [t_fit ones(length(t_fit))]

gamma = fill(NaN, Nmodes - 1)    # growth rate for ky modes 1 … Ny/2-1
for i in 2:Nmodes
    log_amp = log.(frho[i, i_start:i_end])
    all(isfinite, log_amp) || continue
    gamma[i-1] = (X_fit \ log_amp)[1]
end

println("Growth rates γ(ky) fitted over t ∈ [$(t_fit_start), $(t_fit_end)]:")
for (j, g) in enumerate(gamma)
    println("  ky = $(lpad(round(ky_vals[j+1], digits=3), 6))  →  γ = $(round(g, sigdigits=4))")
end

In [ ]:
fig2 = Figure(size=(600, 420))
ax2 = Axis(fig2[1, 1];
           xlabel="ky [ρs⁻¹]",
           ylabel="γ [ω_ci]",
           title="ITG growth rate vs ky  (kx=0, nkz=1)")
scatter!(ax2, ky_vals[2:Nmodes], gamma; markersize=10)
lines!(ax2, ky_vals[2:Nmodes], gamma)
hlines!(ax2, [0.0]; color=:black, linestyle=:dash, linewidth=1)
fig2